# Section 1: Data Verification

| # | Figure | Description |
|---|--------|-------------|
| 1 | Trial inventory | Original vs preprocessed trial counts, missing blocks, split distribution |
| 2 | Trial count summary | Bar chart of DBS-OFF / DBS-ON trial counts per session |
| 3-6 | PSD DBS comparison | Power spectral density per ECoG channel, 4 sessions |
| 7 | Tracing speed DBS comparison | Mean velocity & acceleration traces by DBS condition |

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

In [ ]:
from collections import namedtuple
from pathlib import Path
import numpy as np
import polars as pl
import yaml
import matplotlib.pyplot as plt
from scipy.signal import welch

from modules.style import (
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
    COLOR_DPAD,
    COLOR_PSID,
    apply_modules.style,
    hex_to_rgba,
    panel_label,
    stack_bar_label,
)

apply_modules.style()

In [ ]:
OUT = Path("thesis_figures/sec1")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()
# PSD cells read the raw pre-split parquets — current per-session PSID variants
# only keep a selected subset of neural bands, so the split parquets no longer
# contain all 4 ECoG channels or the Laplacian contacts used by the PSD cells.
raw_data_root = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)
ECOG_CHANNELS = ["ECOG_1", "ECOG_2", "ECOG_3", "ECOG_4"]

In [ ]:
# Session registry — auto-discovered via discover_session_run.
from modules.loaders import (
    discover_session_run,
    EXP_BEHAVIORAL,
    SESSIONS as _SESSION_NAMES,
)

SessionSpec = namedtuple(
    "SessionSpec", ["label", "participant", "session", "psid_variant", "psid_run_ts"]
)

SESSIONS = []
for _name in _SESSION_NAMES:
    _var, _ts = discover_session_run(results_root, "psid", EXP_BEHAVIORAL, _name)
    if not _var:
        continue
    _pid, _sess = _name.split("_S")
    SESSIONS.append(
        SessionSpec(
            label=_name,
            participant=_pid,
            session=int(_sess),
            psid_variant=_var,
            psid_run_ts=_ts,
        )
    )

In [ ]:
from scipy.signal import welch
from scipy.stats import mannwhitneyu
import matplotlib.cm as cm

FS_SPLIT = 200
MARGIN_S = 2
MARGIN_SAMP = MARGIN_S * FS_SPLIT
ECOG_CHANNELS = [1, 2, 3, 4]
LAP_CHANNEL_NAMES = ["8-10", "9-11", "10-12", "11-13", "12-14", "13-15", "14-16"]
raw_data_root = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)


def _session_raw_files(participant, session):
    return sorted(
        (raw_data_root / f"participant_id={participant}" / f"session={session}").glob(
            "*/*.parquet"
        )
    )

## DBS significance heatmap (sessions × frequency)

In [ ]:
# DBS significance heatmap — ECoG (mean across 4 channels) + Laplacian (mean across 7 channels).
from scipy.stats import mannwhitneyu

BAND_ORDER = [
    ("theta_4_8", 4, 8),
    ("alpha_8_12", 8, 12),
    ("beta_12_17", 12, 17),
    ("beta_17_22", 17, 22),
    ("beta_22_27", 22, 27),
    ("beta_27_30", 27, 30),
    ("gamma_30_35", 30, 35),
    ("gamma_35_40", 35, 40),
    ("gamma_40_45", 40, 45),
    ("gamma_45_50", 45, 50),
    ("gamma_50_55", 50, 55),
    ("gamma_55_60", 55, 60),
    ("gamma_60_65", 60, 65),
    ("gamma_70_75", 70, 75),
    ("gamma_75_80", 75, 80),
]


def _compute_band_pvals(channel_prefix):
    n_bands = len(BAND_ORDER)
    pmat = np.full((len(SESSIONS), n_bands), np.nan)
    cols_needed = ["stim"] + [f"{channel_prefix}{bn}_raw" for bn, _, _ in BAND_ORDER]
    for si, s in enumerate(SESSIONS):
        off_powers, on_powers = [], []
        for fp in _session_raw_files(s.participant, s.session):
            df = pl.read_parquet(fp, columns=cols_needed)
            for row in df.iter_rows(named=True):
                cond = "off" if row["stim"] in ("off", "0") else "on"
                powers = []
                for bn, _, _ in BAND_ORDER:
                    arr = np.array(row[f"{channel_prefix}{bn}_raw"], dtype=float)
                    arr = arr[MARGIN_SAMP:-MARGIN_SAMP]
                    powers.append(np.nanmean(arr**2))
                (off_powers if cond == "off" else on_powers).append(powers)
        if len(off_powers) < 2 or len(on_powers) < 2:
            continue
        off_a, on_a = np.array(off_powers), np.array(on_powers)
        for bi in range(n_bands):
            try:
                _, p = mannwhitneyu(off_a[:, bi], on_a[:, bi], alternative="two-sided")
                pmat[si, bi] = -np.log10(max(p, 1e-20))
            except ValueError:
                pass
    return pmat


print("Computing ECoG band p-values...")
_ecog_pmats = [_compute_band_pvals(f"ECOG_{ch}_") for ch in ECOG_CHANNELS]
pmat_ecog = np.nanmean(np.stack(_ecog_pmats), axis=0)

print("Computing Laplacian band p-values...")
_lap_pmats = [
    _compute_band_pvals(f"LAPLACIAN_{name}_LFP_") for name in LAP_CHANNEL_NAMES
]
pmat_lap = np.nanmean(np.stack(_lap_pmats), axis=0)

sess_labels = [s.label for s in SESSIONS]
band_centers = [f"{int((lo + hi) / 2)}" for _, lo, hi in BAND_ORDER]
global_vmax = max(np.nanmax(pmat_ecog), np.nanmax(pmat_lap))

fig, (ax_ecog, ax_lap) = plt.subplots(1, 2, figsize=(10.0, 2.5), layout="constrained")

for ax, pm, letter, title in [
    (ax_ecog, pmat_ecog, "A", "ECoG 1-4"),
    (ax_lap, pmat_lap, "B", "Laplacian LFP 9-16"),
]:
    pm_disp = np.where(np.isfinite(pm), pm, 0)
    im = ax.imshow(
        pm_disp,
        aspect="auto",
        cmap="Greens",
        vmin=0,
        vmax=global_vmax,
        interpolation="nearest",
    )
    ax.set_xticks(range(len(BAND_ORDER)))
    ax.set_xticklabels(band_centers, fontsize=6)
    ax.set_yticks(range(len(sess_labels)))
    ax.set_yticklabels(sess_labels)
    ax.set_xlabel("Frequency (Hz, band center)")
    panel_label(ax, letter, title)
    for si in range(len(SESSIONS)):
        for bi in range(len(BAND_ORDER)):
            val = pm[si, bi]
            if not np.isfinite(val):
                continue
            if val > 3:
                ax.text(
                    bi, si, "***", ha="center", va="center", fontsize=6, color="white"
                )
            elif val > 2:
                ax.text(
                    bi, si, "**", ha="center", va="center", fontsize=6, color="white"
                )
            elif val > -np.log10(0.05):
                ax.text(bi, si, "*", ha="center", va="center", fontsize=6, color="#333")

fig.colorbar(im, ax=ax_lap, label=r"$-\log_{10}(p)$", fraction=0.05, pad=0.02)
fig.savefig(str(OUT / "fig_006_dbs_significance_heatmap.png"))
plt.show()
print("Saved -> fig_006_dbs_significance_heatmap.png")

# envelopes or band powers comparisons.
# significant differences in the power spectra.
# multiple testing (cluster permutation test, mne.)

In [ ]:
# Behavioral DBS separability — tracing_velocity_x & tracing_acceleration_magnitude.
# Per-trial RMS, Mann-Whitney U (DBS-ON vs OFF). Same colorscale as neural heatmap (global_vmax).

BEH_VARS = ["tracing_velocity_x", "tracing_acceleration_magnitude"]
BEH_LABELS = ["$v_x$", "$||a||$"]

beh_pmat = np.full((len(BEH_VARS), len(SESSIONS)), np.nan)  # (vars × sessions)

for si, s in enumerate(SESSIONS):
    fw = s.psid_variant.split("_")[0]
    base = results_root / fw / s.psid_variant / "split"
    rms_by_var = {v: {"off": [], "on": []} for v in BEH_VARS}
    for split_name in ("train", "val", "test"):
        fp = base / f"{split_name}.parquet"
        if not fp.exists():
            continue
        df = pl.read_parquet(fp, columns=["stim"] + BEH_VARS)
        for row in df.iter_rows(named=True):
            cond = "off" if row["stim"] in ("off", "0") else "on"
            for v in BEH_VARS:
                sig = np.array(row[v], dtype=float)
                sig = sig[np.isfinite(sig)]
                if len(sig) < 10:
                    continue
                rms_by_var[v][cond].append(np.sqrt(np.mean(sig**2)))
    for bi, v in enumerate(BEH_VARS):
        off_vals, on_vals = rms_by_var[v]["off"], rms_by_var[v]["on"]
        if len(off_vals) < 2 or len(on_vals) < 2:
            continue
        try:
            _, p = mannwhitneyu(off_vals, on_vals, alternative="two-sided")
            beh_pmat[bi, si] = -np.log10(max(p, 1e-20))
        except ValueError:
            pass

sess_labels = [s.label for s in SESSIONS]
fig, ax = plt.subplots(figsize=(5.0, 2.2), layout="constrained")
pm_disp = np.where(np.isfinite(beh_pmat), beh_pmat, 0)
im = ax.imshow(
    pm_disp,
    aspect="auto",
    cmap="Greens",
    vmin=0,
    vmax=global_vmax,
    interpolation="nearest",
)
ax.set_xticks(range(len(SESSIONS)))
ax.set_xticklabels(sess_labels)
ax.set_yticks(range(len(BEH_VARS)))
ax.set_yticklabels(BEH_LABELS)

for bi in range(len(BEH_VARS)):
    for si in range(len(SESSIONS)):
        val = beh_pmat[bi, si]
        if not np.isfinite(val):
            continue
        if val > 3:
            ax.text(si, bi, "***", ha="center", va="center", fontsize=8, color="white")
        elif val > 2:
            ax.text(si, bi, "**", ha="center", va="center", fontsize=8, color="white")
        elif val > -np.log10(0.05):
            ax.text(si, bi, "*", ha="center", va="center", fontsize=8, color="#333")

fig.colorbar(im, ax=ax, label=r"$-\log_{10}(p)$", fraction=0.06, pad=0.02)
fig.savefig(str(OUT / "fig_006b_beh_dbs_separability.png"))
plt.show()
print("Saved -> fig_006b_beh_dbs_separability.png")

In [ ]:
# check for the outliers, boxplots with each data point.

## Section 3.0 — Bullet points (copy into thesis)

- **Trial counts.** After removing fragmented blocks and plateau trials (cursor stationary > 2 s), usable trial counts were 144 (PDI1\_S2), 116 (PDI1\_S4), 119 (PDI4\_S2), and 120 (PDI4\_S3). Trials were split chronologically into train / val / test sets; DBS-ON and DBS-OFF conditions were balanced in train and test splits for all sessions. Note: PDI4\_S3 validation set contains only DBS-ON trials (0 DBS-OFF) due to the chronological split landing on a pure-ON block.

- **PSD separation.** PDI4\_S2 and PDI4\_S3 show consistent DBS-ON vs DBS-OFF power differences across gamma-band frequencies (30–80 Hz), with Mann-Whitney U tests reaching $p < 0.05$ in multiple bands for both ECoG and Laplacian channels. PDI1\_S2 and PDI1\_S4 show little to no spectral separation between conditions across all bands and channel types.

- **Spectral separability ceiling.** The absence of DBS-modulated signal in PDI1 sessions imposes a hard ceiling on any decoding approach: models cannot recover state information that is not encoded in the neural signal. Chance-level decoding performance on PDI1 throughout all subsequent RQs should be interpreted as a data property rather than a model failure.

- **Behavioral kinematics.** Mean velocity and acceleration traces are visually similar between DBS-ON and DBS-OFF conditions in all four sessions (overlapping SEM bands), consistent with the cursor-tracing task being self-paced and not directly modulated by stimulation at the kinematic level.